In [0]:
%sql
SHOW TABLES IN community_lakehouse.bronze;

database,tableName,isTemporary
bronze,chirps,false
bronze,faostat,false
bronze,unps1,false
bronze,unps2,false


In [0]:
df = spark.read.table("community_lakehouse.bronze.unps1")
df.printSchema()
df.show(3)

root
 |-- t0_hhid: string (nullable = true)
 |-- hhid: string (nullable = true)
 |-- hh_crp2: double (nullable = true)
 |-- hh_crp1: double (nullable = true)
 |-- lvstck: double (nullable = true)
 |-- hh_anm: double (nullable = true)
 |-- hh_plty: double (nullable = true)
 |-- urban: long (nullable = true)
 |-- batch: double (nullable = true)
 |-- hwgt_wc: double (nullable = true)
 |-- region: double (nullable = true)
 |-- wgt: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _source_system: string (nullable = true)

+-------+--------------------+-------+-------+------+------+-------+-----+-----+---------------+------+----------------+--------------------+--------------------+--------------+
|t0_hhid|                hhid|hh_crp2|hh_crp1|lvstck|hh_anm|hh_plty|urban|batch|        hwgt_wc|region|             wgt|        _ingested_at|        _source_file|_source_system|
+-------+--------------------+-------+-------+---

Casting and Checking for validity

In [0]:
from pyspark.sql.functions import col, lit
from pyspark.sql import functions as F

# creating the schema silver
spark.sql("CREATE SCHEMA IF NOT EXISTS community_lakehouse.silver")

# casting certain Yes or No columns to int
df_cast = (df.withColumn("hh_crp2", col("hh_crp2").cast("int"))
            .withColumn("hh_crp1", col("hh_crp1").cast("int"))
            .withColumn("hh_anm", col("hh_anm").cast("int"))
            .withColumn("hh_plty", col("hh_plty").cast("int"))
            .withColumn("urban", col("urban").cast("int")))

# define validity conditions
valid_conditions = [
    F.col("hh_crp2").isin([0, 1]),
    F.col("hh_crp1").isin([0, 1]),
    F.col("hh_anm").isin([0, 1]),
    F.col("hh_plty").isin([0, 1]),
    F.col("urban").isin([0, 1])
]

# filter on valid conditions
df_valid = df_cast.filter(valid_conditions[0])
for cond in valid_conditions[1:]:
    df_valid = df_valid.filter(cond)

# filter out invalid data
df_invalid = df_cast.exceptAll(df_valid)

# adding some metadata
df_valid = df_valid.withColumn("_processed_at", F.current_timestamp())
df_invalid = df_invalid.withColumn("_processed_at", F.current_timestamp())

# write to silver table
df_invalid.write.mode("overwrite").saveAsTable("community_lakehouse.silver.unps1_quarantine")
# write to silver table
df_valid.write.mode("overwrite").saveAsTable("community_lakehouse.silver.unps1")

In [0]:
# select duplicate household_ids
df_duplicates = spark.sql("SELECT t0_hhid FROM community_lakehouse.silver.unps1 GROUP BY t0_hhid HAVING COUNT(t0_hhid) > 1")
display(df_duplicates)

t0_hhid
